In [41]:
!pip install --upgrade pandas


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [51]:
import pandas as pd
import numpy as np

In [52]:
# Que no haya límite en número de columnas
pd.set_option('display.max_columns', None)

# Ajustes opcionales para que no quiebre líneas
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Funciones

## Funciones de preprocesamiento

In [53]:
def replace_long_high_runs(series, threshold=2, min_run=4):
    """
    Replaces stretches of `min_run` or more consecutive values
    above `threshold` with the next value below the threshold.
    """
    s = series.copy()
    i = 0
    n = len(s)

    while i < n:
        if s.iloc[i] > threshold:
            for m in range(i+1,n):
                if s.iloc[m] < threshold:
                    s.iloc[i] = s.iloc[m]
        else:
            i += 1
    return s

## Funciones de agrupamiento

# Dataset shoppers

In [55]:
# Tabla de shoppers sin fine tunning

df_shopper_full = pd.read_csv('Experiments_F_H/checkpoint_shoppers_Dlatent_F/metrics.csv')
df_shopper_full = df_shopper_full[df_shopper_full.method == 'Full-data']

df_shopper_original = pd.read_csv('checkpoint_shoppers_Doriginal_F/metrics_2.csv')
df_shopper_latent = pd.read_csv('checkpoint_shoppers_Dlatent_F/metrics.csv')
df_shopper_latent_ft = pd.read_csv('checkpoint_shoppers_Dlatent_ft_F/metrics.csv')

In [56]:
df_shopper_original.head()

,Unnamed: 0,model,method,cv_balanced,cv_macro_f1,cv_weighted_f1,cv_accuracy,cv_balanced_std,cv_macro_f1_std,cv_weighted_f1_std,cv_accuracy_std,optuna_time,train_time,test_balanced,test_macro_f1,test_weighted_f1,test_accuracy,test_roc_auc,inf_time_per_sample,space,IPC,seed,vae_pretrain_time,vae_fine_tunning_time,encoder_inference_time,decoder_inference_time
0,0,xgb,random,0.900,0.899,0.903,0.91,0.122,0.124,0.120,0.111,11.091,0.026,0.827,0.737,0.838,0.819,0.863,0.000013,original,0.2,5,NaN,NaN,NaN,NaN
1,1,rf,random,0.800,0.792,0.796,0.81,0.100,0.104,0.104,0.097,14.238,0.240,0.738,0.641,0.764,0.730,0.806,0.000017,original,0.2,5,NaN,NaN,NaN,NaN
2,2,svc,random,0.817,0.800,0.800,0.81,0.097,0.103,0.103,0.097,6.370,0.002,0.743,0.695,0.823,0.807,0.841,0.000004,original,0.2,5,NaN,NaN,NaN,NaN
3,3,logreg,random,0.783,0.760,0.760,0.77,0.041,0.033,0.033,0.024,3.906,0.006,0.749,0.650,0.771,0.738,0.815,0.000001,original,0.2,5,NaN,NaN,NaN,NaN
4,4,mlp,random,0.717,0.710,0.713,0.72,0.172,0.170,0.168,0.169,6.784,0.838,0.768,0.635,0.746,0.706,0.836,0.000058,original,0.2,5,NaN,NaN,NaN,NaN


In [57]:
# Preprocesar el valor de IPC que quedo incorrecto para el método random

#df_shopper_original['IPC'] = replace_long_high_runs(df_shopper_original["IPC"])
#df_shopper_original.to_csv('checkpoint_shoppers_Doriginal_F/metrics_2.csv')

In [58]:
metric_col = 'test_balanced'

df_shopper_latent_g = (
    df_shopper_latent
        .groupby(['method', 'model', 'IPC'], as_index=False)
        .agg(**{
            f'{metric_col}_mean': (metric_col, 'mean'),
            f'{metric_col}_std':  (metric_col, 'std')
        })
)



df_shopper_latent_g = df_shopper_latent_g.rename(columns={'IPC':'percentage'})
df_shopper_latent_g = df_shopper_latent_g[df_shopper_latent_g.method == 'k-means']

df_shopper_latent_g.head(20)

,method,model,percentage,test_balanced_mean,test_balanced_std
105,k-means,logreg,0.002,0.8008,0.021653
106,k-means,logreg,0.004,0.8131,0.014910
107,k-means,logreg,0.006,0.8092,0.011924
108,k-means,logreg,0.008,0.8083,0.014166
109,k-means,logreg,0.010,0.8007,0.025456
110,k-means,logreg,0.100,0.8173,0.013030
111,k-means,logreg,0.200,0.8252,0.017054
112,k-means,mlp,0.002,0.7811,0.025375
113,k-means,mlp,0.004,0.7878,0.013028
114,k-means,mlp,0.006,0.7947,0.019630


In [59]:

# --- columna formateada "media ± std" ---
df_shopper_latent_g["result"] = (
    df_shopper_latent_g
      .apply(lambda r: f"{r['test_balanced_mean']:.4f} ± {r['test_balanced_std']:.4f}", axis=1)
)

pivot = (
    df_shopper_latent_g
      .pivot(index="model",
             columns="percentage",
             values=["test_balanced_mean", "test_balanced_std"])
      .swaplevel(0, 1, axis=1)
      .sort_index(axis=1, level=0)
      .rename_axis(index=None, columns=[None, None])
)
pivot = pivot.rename(columns={'test_balanced_mean':'latent_mean',
                            'test_balanced_std':'latent_std'
                             })

display(pivot)

0.002                  0.004                  0.006             \
       latent_mean latent_std latent_mean latent_std latent_mean latent_std   
logreg      0.8008   0.021653      0.8131   0.014910      0.8092   0.011924   
mlp         0.7811   0.025375      0.7878   0.013028      0.7947   0.019630   
rf          0.7759   0.037000      0.7951   0.029380      0.8019   0.029976   
svc         0.8008   0.020660      0.8133   0.021500      0.8138   0.007269   
xgb         0.8084   0.016761      0.8192   0.015010      0.8241   0.006951   

             0.008                  0.010                  0.100             \
       latent_mean latent_std latent_mean latent_std latent_mean latent_std   
logreg      0.8083   0.014166      0.8007   0.025456      0.8173   0.013030   
mlp         0.8008   0.011745      0.7926   0.014630      0.8153   0.011412   
rf          0.8152   0.017080      0.8160   0.016214      0.7333   0.033928   
svc         0.8146   0.017759      0.8189   0.015688      0.8092   0.019668   
xgb         0.8289   0.003929      0.8245   0.008250      0.7247   0.050986   

             0.200             
       latent_mean latent_std  
logreg      0.8252   0.017054  
mlp         0.8041   0.011308  
rf          0.7484   0.024609  
svc         0.7919   0.043534  
xgb         0.7337   0.047404